In [ ]:
# КРОК 1: вхід у Google. Нижче має надрукуватися Ваш акаунт (anastasiia.a.lutsenko@gmail.com)!
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
drive = build('drive', 'v3')
email = drive.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
print('Ви увійшли як:', email)
if email != 'anastasiia.a.lutsenko@gmail.com':
    print('УВАГА: це інший акаунт! Змініть акаунт і запустіть Крок 1 знову.')

In [ ]:
# КРОК 2: копіювання. Можна запускати повторно — готове пропускає.
import time
from googleapiclient.errors import HttpError

SRC_ID = '1UPk8e5sMgvnACPK7HmDGCbfx5D0sWqkS'  # стара папка nrat_pdfs (Sofia)
DST_ID = '1OneRSvCKKVdJNFCHZN3LaGsNhY2D5G42'  # нова папка NRAT (Anastasiia)
FOLDER = 'application/vnd.google-apps.folder'

def list_children(folder_id):
    items, token = [], None
    while True:
        resp = drive.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageSize=1000, pageToken=token).execute()
        items += resp.get('files', [])
        token = resp.get('nextPageToken')
        if not token:
            return items

copied = skipped = 0

def copy_folder(src_id, dst_id, path=''):
    global copied, skipped
    existing = {f['name']: f for f in list_children(dst_id)}
    for item in list_children(src_id):
        name = item['name']
        if item['mimeType'] == FOLDER:
            if name in existing and existing[name]['mimeType'] == FOLDER:
                new_id = existing[name]['id']
            else:
                new_id = drive.files().create(
                    body={'name': name, 'mimeType': FOLDER, 'parents': [dst_id]},
                    fields='id').execute()['id']
            copy_folder(item['id'], new_id, path + '/' + name)
        else:
            if name in existing:
                skipped += 1
                continue
            for attempt in range(5):
                try:
                    drive.files().copy(fileId=item['id'],
                                       body={'name': name, 'parents': [dst_id]}).execute()
                    copied += 1
                    break
                except HttpError:
                    if attempt == 4:
                        raise
                    time.sleep(5 * (attempt + 1))
            if copied % 200 == 0:
                print(f'скопійовано {copied}, пропущено {skipped} — зараз у {path}')

copy_folder(SRC_ID, DST_ID)
print(f'ГОТОВО: скопійовано {copied}, пропущено (вже були) {skipped}')